# INF0093 - Projeto Prático com Sistemas Multiagentes - 2s 2026
## Prof. Marcelo da Silva Reis
## msreis@unicamp.br

# Aula 3 — Do Baseline ao Sistema Agêntico
## Estudo de caso: Assistente de Análise de Editais

Na Aula 2 tínhamos o baseline:

```text
Documento + Pergunta → LLM → Resposta estruturada
```

Hoje construímos a **v2**: um grafo em LangGraph com ferramentas e roteamento condicional.


## Roteiro

1. Configuração e registro da execução.
2. Estrutura herdada da Aula 2: documento, esquema de saída, casos e verificações.
3. Baseline (v1) reimplementado, instrumentado.
4. Ferramentas que realmente leem o documento.
5. Grafo LangGraph com loop agente–ferramentas.
6. Síntese estruturada: a v2 devolve o mesmo esquema da v1.
7. Perguntas simples e compostas; inspeção das *tool calls*.
8. Comparação v1 × v2 na mesma régua.
9. Discussão e ponte para a Aula 4.

## 1. Configuração

In [ ]:
%pip install -q -U langchain langchain-groq langgraph pydantic pandas==2.2.3

In [ ]:
import os, getpass, datetime, platform, time, json

def carregar_chave_groq() -> str:
    """Funciona no Colab (userdata) e localmente (variável de ambiente)."""
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata          # noqa: F401
        os.environ["GROQ_API_KEY"] = userdata.get("SUA_CHAVE_SECRETA_COLAB")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

origem = carregar_chave_groq()
assert os.environ.get("GROQ_API_KEY"), "Chave não configurada."
print("Chave carregada via:", origem)

In [ ]:
from langchain_groq import ChatGroq

# MESMO modelo da Aula 2. Trocar o modelo em v2 invalidaria a comparação:
# não saberíamos se a diferença veio da arquitetura ou do modelo.
#
# MODEL_NAME = "llama-3.3-70b-versatile"   # Meta
MODEL_NAME = "openai/gpt-oss-20b"          # OpenAI

TEMPERATURE = 0
PROMPT_VERSAO = "v2"

llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

RUN_INFO = {
    "modelo": MODEL_NAME,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "arquitetura": "v2-react-tools",
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
}
RUN_INFO

## 2. Estrutura herdada da Aula 2

Documento, esquema de saída, conjunto de casos e funções de verificação são **os mesmos**. Reproduzimos aqui para o notebook ser autocontido; no projeto de vocês, o ideal é importar de um módulo compartilhado ou copiar sem alterar, promovendo assim o reuso de código.

In [ ]:
call_document = """
CHAMADA PARA PROJETOS DE INOVAÇÃO EM SISTEMAS MULTIAGENTES (AGOSTO DE 2026)

OBJETIVO
Apoiar projetos de inovação tecnológica em sistemas multiagentes, com duração
máxima de 12 meses.

ELEGIBILIDADE
Podem submeter propostas:
- pesquisadores vinculados a universidades brasileiras;
- empresas brasileiras em parceria com uma instituição de pesquisa;
- profissionais com cursos de extensão em sistemas multiagentes.

PRAZO
As propostas devem ser submetidas até 30 de outubro de 2026.

DOCUMENTOS OBRIGATÓRIOS
1. Formulário de submissão;
2. Currículo resumido do coordenador;
3. Plano de trabalho;
4. Orçamento estimado.

RESULTADO
O resultado será divulgado até 15 de dezembro de 2026.
"""

print(f"{len(call_document)} caracteres.")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class AnalysisResult(BaseModel):
    answer: str = Field(description="Resposta direta à pergunta.")
    evidence: list[str] = Field(description="Trechos literais do documento que sustentam a resposta.")
    confidence: Literal["high", "medium", "low"]

print("[done]")

In [ ]:
import re, unicodedata

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip()

MARCADORES_AUSENCIA = [
    "nao esta", "nao consta", "nao foi encontrad", "nao encontrei", "nao informad",
    "nao especificad", "nao ha informacao", "nao menciona", "nao e mencionad",
    "ausente no documento", "nao aparece", "nao define", "nao indica",
]

def admite_ausencia(resultado: AnalysisResult) -> bool:
    texto = normalizar(resultado.answer)
    declarou = any(m in texto for m in MARCADORES_AUSENCIA)
    return declarou and len(resultado.evidence) == 0 and resultado.confidence == "low"

def cobertura_esperada(resultado: AnalysisResult, esperado: list) -> float:
    texto = normalizar(resultado.answer)
    return sum(normalizar(k) in texto for k in esperado) / len(esperado)

def evidencia_fiel(resultado: AnalysisResult, documento: str) -> Optional[float]:
    if not resultado.evidence:
        return None
    doc = normalizar(documento)
    validas = [e for e in resultado.evidence if normalizar(e) in doc]
    return len(validas) / len(resultado.evidence)

def avaliar(caso: dict, resultado: AnalysisResult) -> dict:
    if caso["verificacao"] == "manual":
        return {"aprovado": None, "cobertura": None}
    if caso["esperado"] is None:
        return {"aprovado": admite_ausencia(resultado), "cobertura": None}
    cobertura = cobertura_esperada(resultado, caso["esperado"])
    return {"aprovado": cobertura >= caso.get("cobertura_minima", 1.0),
            "cobertura": round(cobertura, 2)}

print("[done]")

### O conjunto congelado, mais um caso composto

Os cinco casos da Aula 2 permanecem **inalterados** — eles são a régua.

Acrescentamos T06, uma pergunta composta. Note a assimetria: um caso novo pode ser adicionado, mas
os antigos não podem ser mexidos, e o baseline precisa ser reexecutado sobre o conjunto completo
(é o que faremos na seção 8).

In [ ]:
test_cases = [
    {"id": "T01", "tipo": "normal", "verificacao": "auto",
     "pergunta": "Qual é o prazo para submissão?",
     "esperado": ["30 de outubro de 2026"], "cobertura_minima": 1.0},

    {"id": "T02", "tipo": "lista", "verificacao": "auto",
     "pergunta": "Quais documentos são obrigatórios?",
     "esperado": ["formulário", "currículo", "plano de trabalho", "orçamento"],
     "cobertura_minima": 1.0},

    {"id": "T03", "tipo": "interpretação", "verificacao": "auto",
     "pergunta": "Quem pode participar?",
     "esperado": ["universidades brasileiras", "empresas brasileiras"],
     "cobertura_minima": 0.5},

    {"id": "T04", "tipo": "informação ausente", "verificacao": "auto",
     "pergunta": "Qual é o valor máximo de financiamento?",
     "esperado": None},

    {"id": "T05", "tipo": "ambíguo", "verificacao": "manual",
     "pergunta": "Qual é o prazo?",
     "esperado": None,
     "nota": "Esperado: distinguir submissão (30/10) de resultado (15/12) ou pedir esclarecimento."},

    # NOVO na Aula 3: é a limitação que motiva a v2.
    #
    {"id": "T06", "tipo": "composto", "verificacao": "auto",
     "pergunta": "Quem pode participar e qual é o prazo para submissão?",
     "esperado": ["universidades brasileiras", "30 de outubro de 2026"],
     "cobertura_minima": 1.0},
]

print(len(test_cases), "casos;",
      sum(c["verificacao"] == "auto" for c in test_cases), "automáticos.")

## 3. Baseline (v1) instrumentado

Idêntico ao da Aula 2. Ele será reexecutado para que a comparação use números coletados **hoje**,
com o mesmo modelo e no mesmo ambiente.

In [ ]:
SYSTEM_V1 = """
Você é um assistente de análise documental.
Responda exclusivamente com base no documento.
Não invente informações.
Se a informação não estiver presente, a 'answer' deve dizer explicitamente que a informação não
consta no documento, a 'evidence' deve ser uma lista vazia e a 'confidence' deve ser 'low'.
Forneça em 'evidence' apenas trechos copiados literalmente do documento.
Use confiança 'high' para informação explícita, 'medium' para interpretação e 'low' para incerteza.
"""

structured_llm = llm.with_structured_output(AnalysisResult, include_raw=True)

def responder_v1(pergunta: str, documento: str = call_document):
    """Baseline da Aula 2: uma chamada ao LLM."""
    prompt = SYSTEM_V1 + "\n\nDOCUMENTO:\n" + documento + "\n\nPERGUNTA:\n" + pergunta

    inicio = time.perf_counter()
    saida = structured_llm.invoke(prompt)
    latencia = time.perf_counter() - inicio

    uso = getattr(saida["raw"], "usage_metadata", None) or {}
    metricas = {
        "latencia_s": round(latencia, 2),
        "tokens_entrada": uso.get("input_tokens"),
        "tokens_saida": uso.get("output_tokens"),
        "chamadas_llm": 1,
        "chamadas_tool": 0,
        "erros_tool": 0,
    }
    return saida["parsed"], metricas

print("[done]")

## 4. Ferramentas

Três cuidados que valem para o projeto de vocês:

1. **A ferramenta precisa fazer trabalho de verdade.** Se ela devolve uma resposta fixa, o
   "sistema agêntico" é teatro: funcionaria com o documento errado. As nossas leem o documento.
2. **A docstring é prompt.** É por ela que o modelo decide se e quando chamar a ferramenta. Escreva
   pensando em quem vai ler: o LLM.
3. **A ferramenta pode falhar.** Devolver uma mensagem de erro tratável é melhor que levantar
   exceção no meio do grafo.

In [ ]:
from langchain_core.tools import tool

def _secao(documento: str, titulo: str) -> str:
    """Extrai uma seção do edital pelo cabeçalho em maiúsculas."""
    linhas = documento.strip().split("\n")
    capturando, coletado = False, []
    for linha in linhas:
        cabecalho = linha.strip().isupper() and len(linha.strip()) > 3
        if cabecalho:
            if capturando:
                break
            capturando = normalizar(titulo) in normalizar(linha)
            continue
        if capturando and linha.strip():
            coletado.append(linha.strip())
    return "\n".join(coletado)

@tool
def consultar_prazo() -> str:
    """Devolve o trecho do edital que trata de prazos de submissão. Use para perguntas sobre datas
    de envio de propostas."""
    trecho = _secao(call_document, "PRAZO")
    return trecho or "Seção de prazo não encontrada no documento."

@tool
def consultar_elegibilidade() -> str:
    """Devolve o trecho do edital que trata de quem pode submeter propostas. Use para perguntas
    sobre elegibilidade, participação ou requisitos do proponente."""
    trecho = _secao(call_document, "ELEGIBILIDADE")
    return trecho or "Seção de elegibilidade não encontrada no documento."

@tool
def consultar_documentos() -> str:
    """Devolve a lista de documentos obrigatórios exigidos pelo edital. Use para perguntas sobre
    anexos, formulários ou documentação da submissão."""
    trecho = _secao(call_document, "DOCUMENTOS OBRIGATÓRIOS")
    return trecho or "Seção de documentos não encontrada no documento."

tools = [consultar_prazo, consultar_elegibilidade, consultar_documentos]

# Confira que elas realmente leem o documento:
#
print(consultar_prazo.invoke({}))
print("---")
print(consultar_elegibilidade.invoke({}))
print("---")
print(consultar_documentos.invoke({}))

## 5. Grafo LangGraph

O estado carrega as mensagens (com o *reducer* `add_messages`, que **acumula** em vez de
sobrescrever; só lembrando que *reducer* é uma função no LangGraph que define como combinar o valor atual de um campo do estado com um novo valor), o documento e um contador de chamadas a ferramentas.

O `recursion_limit` é o freio: sem ele, um agente que insiste em chamar ferramentas roda até
estourar cota. Todo sistema agêntico precisa de uma condição de parada explícita.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


class State(TypedDict):
    messages: Annotated[list, add_messages]  # usa o reducer add_messages.
    document: str


SYSTEM_V2 = """
Você é um assistente de análise de editais.
Use as ferramentas disponíveis para obter os trechos do documento de que precisa.
Para perguntas compostas, consulte TODAS as informações necessárias antes de responder.
Não invente informações: se algo não estiver no documento, diga isso explicitamente.
"""


# Ligando as ferramentas ao LLM.
#
llm_with_tools = llm.bind_tools(tools)


# Todo nó agêntico é instruído com a mensagem de sistema mais o histórico de
# mensagens de estado.
#
def agent_node(state: State):
    mensagens = [SystemMessage(content=SYSTEM_V2)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(mensagens)]}


# Estes comandos definem a arquitetura do grafo (nós e arestas).
#
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)   # -> "tools" ou END
builder.add_edge("tools", "agent")

app = builder.compile()

print("[grafo compilado]")

In [ ]:
# Visualização do grafo (opcional, requer IPython).
#
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as erro:
    print("Sem renderização de imagem:", type(erro).__name__)
    print(app.get_graph().draw_ascii())

## 6. Síntese estruturada

O loop agente–ferramentas devolve texto livre. Para comparar com a v1 precisamos do **mesmo
esquema** (`answer`, `evidence`, `confidence`), então acrescentamos uma etapa final de formatação.

Isso custa uma chamada a mais ao LLM — e esse custo aparecerá na tabela de comparação. É
exatamente o tipo de troca que o Entregável 2 pede para explicitar.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

SYSTEM_FORMAT = """
Converta a análise abaixo no formato estruturado.
'evidence' deve conter apenas trechos copiados literalmente do documento.
Se a informação não constar no documento, a 'answer' deve dizer isso explicitamente,
'evidence' deve ser lista vazia e 'confidence' deve ser 'low'.
"""

def responder_v2(pergunta: str, documento: str = call_document, limite: int = 10):
    """v2: loop agente-ferramentas em LangGraph + síntese estruturada."""

    inicio = time.perf_counter()  # Início da contagem de tempo (p/ latência).

    # Consulta ao agente com uma pergunta, o documento e um limite de iterações
    # para o loop agente-tools.
    #
    estado = app.invoke(
        {"messages": [HumanMessage(content=pergunta)], "document": documento},
        config={"recursion_limit": limite},
    )

    mensagens = estado["messages"]

    # Conta o número de tool calls.
    #
    chamadas_tool = sum(len(getattr(m, "tool_calls", None) or []) for m in mensagens)

    # Conta quantas mensagens de ferramenta (ToolMessage) representam erros.
    #
    erros_tool = sum(
        1 for m in mensagens
        if isinstance(m, ToolMessage) and getattr(m, "status", None) == "error"
    )

    # Conta o número de chamadas do LLM (AIMessage).
    #
    chamadas_llm = sum(1 for m in mensagens if isinstance(m, AIMessage))

    # Transcreve as mensagens em um string.
    #
    transcricao = "\n".join(
        f"{type(m).__name__}: {m.content}" for m in mensagens if getattr(m, "content", "")
    )

    # Pede para o LLM gerar a saída estruturada.
    #
    saida = structured_llm.invoke(
        SYSTEM_FORMAT + "\n\nDOCUMENTO:\n" + documento
        + "\n\nPERGUNTA:\n" + pergunta
        + "\n\nANÁLISE:\n" + transcricao
    )

    latencia = time.perf_counter() - inicio   # Cálculo da latência.

    # Dict saida["raw"] contém AIMessage, que por sua vez contém a resposta
    # bruta do LLM e também um dict de meta-dados da invocação.
    #
    uso = getattr(saida["raw"], "usage_metadata", None) or {}

    # Guardando as métricas do uso do agente + chamada adicional do LLM.
    #
    metricas = {
        "latencia_s": round(latencia, 2),
        "tokens_entrada": uso.get("input_tokens"),
        "tokens_saida": uso.get("output_tokens"),
        "chamadas_llm": chamadas_llm + 1,      # + a chamada de formatação
        "chamadas_tool": chamadas_tool,
        "erros_tool": erros_tool,
    }

    # Devolve a saída estruturada igual ao baseline, mais as métricas do uso
    # total e as mensagens de estado do agente.
    #
    return saida["parsed"], metricas, mensagens

print("[done]")

> **Sobre as métricas de token:** `tokens_entrada` e `tokens_saida` acima contam apenas a chamada
> de formatação, porque só dela guardamos a mensagem bruta. Somar o consumo de todo o grafo exige
> percorrer o `usage_metadata` de cada `AIMessage` — fica como exercício, e é um bom exemplo de
> como a instrumentação fica mais difícil à medida que a arquitetura cresce.

## 7. Pergunta simples e pergunta composta

In [ ]:
resultado, metricas, _ = responder_v2("Qual é o prazo para submissão?")
print("RESPOSTA :", resultado.answer)
print("EVIDÊNCIA:", resultado.evidence)
print("MÉTRICAS :", metricas)

In [ ]:
resultado, metricas, mensagens = responder_v2("Quem pode participar e qual é o prazo para submissão?")
print("RESPOSTA :", resultado.answer)
print("MÉTRICAS :", metricas)

### Inspecionar o que aconteceu por dentro

Observabilidade é parte da arquitetura: sem olhar as mensagens intermediárias, não há como saber
se o agente usou a ferramenta certa, usou ferramenta demais, ou respondeu de memória.

Observe que o nosso histórico de mensagens de estado inclui mensagens do usuário (`HumanMessage`), do LLM (`AIMessage`) e das ferramentas (`ToolMessage`).


In [ ]:
for i, m in enumerate(mensagens):

    print("=" * 78)

    print(i+1, type(m).__name__)

    # Imprime a ferramenta chamada se a saída do AIMessage indicar que o LLM
    # quis fazer uma tool call na iteração em questão.
    #
    if getattr(m, "tool_calls", None):
        print("TOOL CALLS:", [(tc["name"], tc["args"]) for tc in m.tool_calls])

    # Imprime o conteúdo da mensagem (truncando em 400 caracteres, para não
    # ocupar muito espaço na nossa tela).
    #
    conteudo = getattr(m, "content", "")
    print(conteudo[:400] + ("..." if len(conteudo) > 400 else ""))

print("=" * 78)

## 8. Comparação v1 × v2

Mesmo documento, mesmos casos, mesmo modelo, mesmo esquema de saída, mesmas funções de verificação.
A única variável que muda é a arquitetura.

In [ ]:
registros = []

for caso in test_cases:
    r1, m1 = responder_v1(caso["pergunta"])
    a1 = avaliar(caso, r1)

    r2, m2, _ = responder_v2(caso["pergunta"])
    a2 = avaliar(caso, r2)

    registros.append({
        "id": caso["id"], "tipo": caso["tipo"],
        "v1_aprovado": a1["aprovado"], "v2_aprovado": a2["aprovado"],
        "v1_cobertura": a1["cobertura"], "v2_cobertura": a2["cobertura"],
        "v1_evid": evidencia_fiel(r1, call_document),
        "v2_evid": evidencia_fiel(r2, call_document),
        "v1_latencia": m1["latencia_s"], "v2_latencia": m2["latencia_s"],
        "v1_llm": m1["chamadas_llm"], "v2_llm": m2["chamadas_llm"],
        "v2_tools": m2["chamadas_tool"], "v2_erros_tool": m2["erros_tool"],
        "v1_resposta": r1.answer, "v2_resposta": r2.answer,
    })

    print(f'[{caso["id"]}] v1={a1["aprovado"]}  v2={a2["aprovado"]}  '
          f'(tools: {m2["chamadas_tool"]}, {m1["latencia_s"]}s -> {m2["latencia_s"]}s)')


In [ ]:
import pandas as pd

df = pd.DataFrame(registros)
df[["id", "tipo", "v1_aprovado", "v2_aprovado", "v1_latencia", "v2_latencia",
    "v1_llm", "v2_llm", "v2_tools"]]

In [ ]:
autos = df[df["v1_aprovado"].notna()]

COMPARACAO = {
    "casos_automaticos": int(len(autos)),
    "v1_taxa_aprovacao": round(float(autos["v1_aprovado"].astype(bool).mean()), 2),
    "v2_taxa_aprovacao": round(float(autos["v2_aprovado"].astype(bool).mean()), 2),
    "v1_latencia_mediana_s": round(float(df["v1_latencia"].median()), 2),
    "v2_latencia_mediana_s": round(float(df["v2_latencia"].median()), 2),
    "v1_chamadas_llm": int(df["v1_llm"].sum()),
    "v2_chamadas_llm": int(df["v2_llm"].sum()),
    "v2_chamadas_tool": int(df["v2_tools"].sum()),
    "v2_erros_tool": int(df["v2_erros_tool"].sum()),
}
COMPARACAO

### Como ler esses números

Com 5 casos automáticos, uma diferença de acerto vale 20 pontos percentuais. Um resultado
como "v1 = 0,80 e v2 = 1,00" significa um caso, não uma tendência.

O que se pode afirmar com honestidade:

- se a v2 acerta o caso composto (T06) e a v1 não, isso é evidência direta da hipótese arquitetural;
- o aumento de latência e de chamadas ao LLM é medido, não estimado;
- para afirmar que "a v2 é melhor" de forma geral, seria preciso um conjunto bem maior.

**Se a v2 for pior ou empatar, isso também é resultado**, e deve ser relatado, com hipótese sobre a causa (por exemplo, o problema ser suficientemente simples para que a arquitetura simplificada seja o bastante para resolver o problema de forma adequada).

In [ ]:
referencia = {"run": RUN_INFO, "comparacao": COMPARACAO, "registros": registros}

with open("v2_vs_baseline_resultados.json", "w", encoding="utf-8") as f:
    json.dump(referencia, f, ensure_ascii=False, indent=2, default=str)

print("Salvo em v2_vs_baseline_resultados.json")

## 9. Discussão

1. A v2 respondeu melhor às perguntas compostas?
2. A v2 usou ferramentas quando deveria — e **evitou** usá-las quando não precisava?
3. Quanto custou a autonomia, em latência e em chamadas ao LLM?
4. Surgiram modos de falha que a v1 não tinha (ferramenta errada, loop, formato quebrado)?
5. O que aconteceria se o documento tivesse 100 páginas e as ferramentas precisassem buscar trechos?
6. Que parte desta arquitetura seria candidata a virar um agente especializado no Entregável 3?

### Ponte para a Aula 4

A v2 sabe **agir**, mas ainda não sabe **lembrar**: cada pergunta começa do zero. Uma pergunta de
acompanhamento como *"e qual é o prazo para eles?"* não tem a quem se referir.

Aula 4: memória e checkpoints; e MCP como fronteira de integração.